# 5. Architektura Transformer

## 1. Dane wejściowe

In [1]:
import torch

SEED = 42
torch.manual_seed(SEED)
torch.set_printoptions(precision=4, sci_mode=False)

batch_size = 2
n = 3
d_model = 4

X = torch.randn(batch_size, n, d_model)

print("Seed:", SEED)
print("Kształt X:", tuple(X.shape))
print("X:\n", X)
print("Pierwsza sekwencja:\n", X[0])
print("Pierwszy token pierwszej sekwencji:", X[0, 0])

Seed: 42
Kształt X: (2, 3, 4)
X:
 tensor([[[ 1.9269,  1.4873,  0.9007, -2.1055],
         [ 0.6784, -1.2345, -0.0431, -1.6047],
         [ 0.3559, -0.6866, -0.4934,  0.2415]],

        [[-1.1109,  0.0915, -2.3169, -0.2168],
         [-0.3097, -0.3957,  0.8034, -0.6216],
         [-0.5920, -0.0631, -0.8286,  0.3309]]])
Pierwsza sekwencja:
 tensor([[ 1.9269,  1.4873,  0.9007, -2.1055],
        [ 0.6784, -1.2345, -0.0431, -1.6047],
        [ 0.3559, -0.6866, -0.4934,  0.2415]])
Pierwszy token pierwszej sekwencji: tensor([ 1.9269,  1.4873,  0.9007, -2.1055])


### 1.1 Projekcje Q, K, V

In [2]:
d_k = 2
d_v = 3

W_Q = torch.randn(d_model, d_k)
W_K = torch.randn(d_model, d_k)
W_V = torch.randn(d_model, d_v)

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print("Kształt W_Q:", tuple(W_Q.shape))
print("Kształt W_K:", tuple(W_K.shape))
print("Kształt W_V:", tuple(W_V.shape))
print("Kształt Q:", tuple(Q.shape))
print("Kształt K:", tuple(K.shape))
print("Kształt V:", tuple(V.shape))
print("W_Q:\n", W_Q)
print("W_K:\n", W_K)
print("W_V:\n", W_V)
print("Pierwszy token x:", X[0, 0])
print("Jego zapytanie q:", Q[0, 0])
print("Jego klucz k:", K[0, 0])
print("Jego wartość v:", V[0, 0])

Kształt W_Q: (4, 2)
Kształt W_K: (4, 2)
Kształt W_V: (4, 3)
Kształt Q: (2, 3, 2)
Kształt K: (2, 3, 2)
Kształt V: (2, 3, 3)
W_Q:
 tensor([[ 1.3525,  0.6863],
        [-0.3278,  0.7950],
        [ 0.2815,  0.0562],
        [ 0.5227, -0.2384]])
W_K:
 tensor([[-0.0499,  0.5263],
        [-0.0085,  0.7291],
        [ 0.1331,  0.8640],
        [-1.0157, -0.8887]])
W_V:
 tensor([[ 0.1498, -0.2089, -0.3870],
        [ 0.9912,  0.4679, -0.2049],
        [-0.7409,  0.3618,  1.9199],
        [-0.2254, -0.3417,  0.3040]])
Pierwszy token x: tensor([ 1.9269,  1.4873,  0.9007, -2.1055])
Jego zapytanie q: tensor([1.2718, 3.0573])
Jego klucz k: tensor([2.1496, 4.7480])
Jego wartość v: tensor([1.5701, 1.3388, 0.0386])


## 2. Funkcja pojedynczej głowy attention i weryfikacja kształtów

In [10]:
def attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = (Q @ K.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        scores = scores + mask
    if torch.isneginf(scores).all(dim=-1).any():
        raise ValueError("Każde zapytanie musi mieć przynajmniej jeden dozwolony klucz.")
    weights = torch.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights


output, weights = attention(Q, K, V)

print("Kształt Q:", tuple(Q.shape))
print("Kształt K:", tuple(K.shape))
print("Kształt V:", tuple(V.shape))
print("Kształt wag atencji:", tuple(weights.shape))
print("Kształt wyjścia:", tuple(output.shape))
print("Czy d_v różni się od d_k:", V.size(-1) != Q.size(-1))

Kształt Q: (2, 3, 2)
Kształt K: (2, 3, 2)
Kształt V: (2, 3, 3)
Kształt wag atencji: (2, 3, 3)
Kształt wyjścia: (2, 3, 3)
Czy d_v różni się od d_k: True


## 3. Przykład z trzema pozycjami i jednostkowymi projekcjami

In [13]:
X_example = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0]
])

W_Q_example = torch.eye(2)
W_K_example = torch.eye(2)
W_V_example = torch.eye(2)

Q_example = X_example @ W_Q_example
K_example = X_example @ W_K_example
V_example = X_example @ W_V_example

output_example, weights_example = attention(Q_example, K_example, V_example)
scores_example = (Q_example @ K_example.transpose(-2, -1)) / (Q_example.size(-1) ** 0.5)

print("Dopasowania (scores):\n", scores_example)
print("Wagi (weights):\n", weights_example)
print("Wyjście (output):\n", output_example)

Dopasowania (scores):
 tensor([[0.7071, 0.0000, 0.7071],
        [0.0000, 0.7071, 0.7071],
        [0.7071, 0.7071, 1.4142]])
Wagi (weights):
 tensor([[0.4011, 0.1978, 0.4011],
        [0.1978, 0.4011, 0.4011],
        [0.2483, 0.2483, 0.5035]])
Wyjście (output):
 tensor([[0.8022, 0.5989],
        [0.5989, 0.8022],
        [0.7517, 0.7517]])


## 4. Maska przyczynowa

In [ ]:
causal_mask = torch.triu(
    torch.full((n, n), float("-inf"), dtype=Q.dtype, device=Q.device),
    diagonal=1,
)
causal_output, causal_weights = attention(Q, K, V, mask=causal_mask)

print("Maska przyczynowa:\n", causal_mask)
print("Wagi z maską przyczynową (pierwsza sekwencja):\n", causal_weights[0])

X_mod = X.clone()
X_mod[:, -1, :] = torch.randn(batch_size, d_model)
Q_mod = X_mod @ W_Q
K_mod = X_mod @ W_K
V_mod = X_mod @ W_V

causal_output_mod, causal_weights_mod = attention(Q_mod, K_mod, V_mod, mask=causal_mask)

print("Wagi po modyfikacji ostatniego tokenu:\n", causal_weights_mod[0])
print("Wyjście przed modyfikacją:\n", causal_output[0])
print("Wyjście po modyfikacji:\n", causal_output_mod[0])

Maska przyczynowa:
 tensor([[0., -inf, -inf],
        [0., 0., -inf],
        [0., 0., 0.]])
Wagi z maską przyczynową (pierwsza sekwencja):
 tensor([[1.0000, 0.0000, 0.0000],
        [0.4522, 0.5478, 0.0000],
        [0.2157, 0.4791, 0.3052]])
Wyjście pierwszej pozycji: tensor([1.5701, 1.3388, 0.0386])
Wartość V pierwszej pozycji: tensor([1.5701, 1.3388, 0.0386])
Wagi po modyfikacji ostatniego tokenu:
 tensor([[1.0000, 0.0000, 0.0000],
        [0.4522, 0.5478, 0.0000],
        [0.3752, 0.2538, 0.3710]])
Wyjście przed modyfikacją:
 tensor([[ 1.5701,  1.3388,  0.0386],
        [ 0.3110,  0.5032, -0.3003],
        [-0.1069, -0.0011, -0.5354]])
Wyjście po modyfikacji:
 tensor([[ 1.5701,  1.3388,  0.0386],
        [ 0.3110,  0.5032, -0.3003],
        [ 1.2470,  0.6130, -1.4201]])


## 5. Maska paddingu

In [17]:
pad_mask = torch.zeros(batch_size, 1, n)
pad_mask[0, :, 2] = float("-inf")

pad_output, pad_weights = attention(Q, K, V, mask=pad_mask)

print("Maska paddingu dla pierwszej sekwencji:\n", pad_mask[0])
print("Wagi pierwszej sekwencji z maską paddingu:\n", pad_weights[0])
print("Suma wag po kluczach (pierwsza sekwencja):\n", pad_weights[0].sum(dim=-1))

X_pad_mod = X.clone()
X_pad_mod[0, 2, :] = torch.randn(d_model)
Q_pad_mod = X_pad_mod @ W_Q
K_pad_mod = X_pad_mod @ W_K
V_pad_mod = X_pad_mod @ W_V

pad_output_mod, _ = attention(Q_pad_mod, K_pad_mod, V_pad_mod, mask=pad_mask)

print("Wyjście poprawnych zapytań przed zmianą tokenu paddingu:\n", pad_output[0, :2])
print("Wyjście poprawnych zapytań po zmianie tokenu paddingu:\n", pad_output_mod[0, :2])

Maska paddingu dla pierwszej sekwencji:
 tensor([[0., 0., -inf]])
Wagi pierwszej sekwencji z maską paddingu:
 tensor([[0.9999, 0.0001, 0.0000],
        [0.4522, 0.5478, 0.0000],
        [0.3104, 0.6896, 0.0000]])
Suma wag po kluczach (pierwsza sekwencja):
 tensor([1., 1., 1.])
Wyjście poprawnych zapytań przed zmianą tokenu paddingu:
 tensor([[ 1.5698,  1.3385,  0.0386],
        [ 0.3110,  0.5032, -0.3003]])
Wyjście poprawnych zapytań po zmianie tokenu paddingu:
 tensor([[ 1.5698,  1.3385,  0.0386],
        [ 0.3110,  0.5032, -0.3003]])


## Komentarz

Softmax działa po osi kluczy, maska modyfikuje wartości dopasowań przed softmaxem, liczba cech wyjścia zalezy od V, bo to iloczyn wag i V.  

## Wykorzystanie AI

- Narzędzie: OpenCode (OpenChamber), model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol).
- Zakres: przygotowanie tensora wejściowego i projekcji Q, K, V.
- Narzędzie: Antigravity, model: Gemini 3.8 Flash.
- Zakres: uproszczenie sekcji 2 zgodnie z punktami 2 i 3 zadania, przygotowanie sekcji 3 z przykładem liczbowym (punkt 4 zadania), aktualizacja sekcji 4 z bezpośrednim wypisywaniem wag i wyjść maski przyczynowej, przygotowanie sekcji 5 z maską paddingu (punkt 6 zadania) oraz osadzenie treści skilla zajecia-nlp w sekcji promptów.
- Historia promptów oraz treść użytego skilla znajdują się w sekcji poniżej oraz w [used_prompts.md](used_prompts.md).

## Used prompts

### Użyty skill (zajecia-nlp)

```markdown
---
name: zajecia-nlp
description: >
  Guides student-AI collaboration on weekly Jupyter tasks for Uczenie głębokie
  w przetwarzaniu tekstu (Christop, AMU) from the live course page, prioritizing
  understanding and student-led execution. Keeps a full prompt log,
  a Wykorzystanie AI section, and ASCII hyphens with straight quotes.
  Use when editing a course notebook,
  used_prompts.md, a chapter task, tokenization, BoW, BPE, or a submission
  in this repo.
---

# Zajęcia NLP

Task text and dates live on the course page. Fetch them; do not rely on a copied chapter.

https://christop.faculty.wmi.amu.edu.pl/index.php?page=course&id=uczenie-glebokie-w-przetwarzaniu-tekstu

Chapter: `index.php?page=chapter&id=uczenie-glebokie-w-przetwarzaniu-tekstu&topic=<slug>`.

Organizational rules: the same URL with `topic=informacje-organizacyjne`.

Folder `N/` is week N. `N/N.ipynb` is the solution. `N/used_prompts.md` is the prompt log.

## Collaboration

The goal is the student's understanding of the topic, not autonomous completion of the submission. Work on one agreed part at a time. Discuss the reasoning, tensor shapes, or expected behavior in chat as relevant to that part, while keeping notebook prose limited to the requested explanations.

Let the student run the code and interpret the results. When useful, ask them to predict an outcome before running it. Use their outputs and questions to explain the mechanism and agree on the next step. A link to the chapter alone is a starting point for discussing scope, not permission to solve the entire assignment.

## Execution

Run notebook cells, training, tests, validation scripts (including `check_notebook.py`), or other automated checks only when the student explicitly asks you to run them. Permission to prepare or change code is not permission to execute it. Otherwise, prepare runnable code and checks for the student, state what to run next and what to observe, and wait for their results. Preserve existing outputs; never invent results or claim unrun checks passed.

## Start

1. Fetch the chapter for this week. Read its **Zadanie** section. Fetch the organizational chapter when a deadline or submission rule is in play.
2. Read `N/N.ipynb` and `N/used_prompts.md`.
3. Restate the requested slice in one short paragraph: inputs, outputs, and the notebook heading it belongs under. Write code after that restatement follows the student's own request. When the message is only a paste of the chapter task, ask them to say what they want done before writing code.

## Implement

4. Change only the requested part. Keep every part of the week's task in the same notebook when the chapter says so.
5. Leave runnable code; the student saves outputs after execution unless they explicitly ask you to run it. Add markdown prose or a comment only where the chapter or the student asks for an explanation, analysis, or comment. Otherwise a section gets its heading alone.
6. Use markdown headings so each part of the task is easy to find.
7. Record the seed when the task asks for an experiment.
8. Fit vocabularies, tokenizers, and model selection on the training and validation splits. Score the chosen checkpoint once on the test split.

## Before you finish

9. Append this turn's user prompt to `N/used_prompts.md`, in send order, including follow-ups. Keep every earlier prompt. Record the tool and model name when they are known.
10. Add or update a notebook section **Wykorzystanie AI**: tool, model name when known, what it did, and a pointer to `used_prompts.md`.
11. In descriptions, conclusions, comments, and README, use the ASCII hyphen `-` and straight quotes `"`. Leave mathematical symbols, characters that belong to the data, and the prompt log as written.
12. For typography validation, offer `python3 .cursor/skills/zajecia-nlp/scripts/check_notebook.py N/N.ipynb` for the student to run. Apply the execution rule above to running or rerunning it yourself.
13. End the turn when the agreed part is prepared, the prompt log includes this turn, and **Wykorzystanie AI** points at that log. Report which results were actually observed and which checks remain unrun. If execution is pending, give the student a focused next step rather than claiming the submission is verified.

## Hand-in

A class task is one `.ipynb` with code, outputs, and the written analysis the chapter asks for. Before hand-in, the student runs the notebook from the top, saves the outputs, and performs the typography check, or explicitly asks the AI to do those steps. Discuss observed results together before writing conclusions.
```

### Historia promptów

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
/zajecia-nlp https://christop.faculty.wmi.amu.edu.pl/index.php?page=chapter&id=uczenie-glebokie-w-przetwarzaniu-tekstu&topic=architektura-transformer
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
ok
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
zmien skill aby nie uruchamial automatycznie testow, ma polegac on na wspolpracy miedzy studentem a AI i celem jest zrozumienie zagadnienia
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
ok wiec zresetuj teraz i dawaj
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
Odpowiedź na pytanie o zakres resetu: Notebook 5 od początku
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
nie mam pojcia, ale uruchomilem komorke
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
2x [x, x, x], gdzie x = [c, c, c, c]?
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
ok, co dalej?
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
ok
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
pierwszy?
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
Kształt K po transpozycji: (2, 2, 3)
Kształt dopasowań: (2, 3, 3)
Iloczyny skalarne dla pierwszej sekwencji:
 tensor([[17.2498,  4.6220, -3.3278],
        [ 0.3686,  0.6396, -0.0227],
        [-0.3457,  0.7831,  0.1452]])
Skalowane dopasowania dla pierwszej sekwencji:
 tensor([[12.1975,  3.2683, -2.3531],
        [ 0.2607,  0.4523, -0.0160],
        [-0.2444,  0.5538,  0.1026]])
Dopasowania zapytania pierwszego tokenu do trzech kluczy: tensor([12.1975,  3.2683, -2.3531])
Iloczyn q_0 i k_0 policzony osobno: tensor(17.2498)
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
Ok wygląda prawidłowego
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
ok
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
oki
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
dodaj used prompts do aktualnego pliku na koncu
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
oraz sprawdz notebook
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
chdozilo mi pythonem
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
w skillu jest tscrpt
```

```text
tool: Antigravity
model: Gemini 3.8 Flash
continue on 5 where we have finished
```

```text
tool: Antigravity
model: Gemini 3.8 Flash
i think we're overcomplicating the 2nd section. Should take way less code blocks. We already have 5 where most of the code is printing stuff. Please fall back to the notes and let's redo this part: Zaimplementuj skalowane iloczyny zapytań i kluczy, opcjonalną maskę dodawaną przed softmaxem, softmax po osi kluczy oraz mnożenie przez 
V
V. Możesz używać podstawowych operacji tensorowych i gotowego softmaxu. Zwracaj zarówno wyjście, jak i wagi atencji.
Wypisz kształty - 
Q
Q i 
K
K mają 
batch
×
n
×
d
k
batch×n×d 
k

 , 
V
V ma 
batch
×
n
×
d
v
batch×n×d 
v

 , wagi mają 
batch
×
n
×
n
batch×n×n, a wyjście 
batch
×
n
×
d
v
batch×n×d 
v

 . Transponuj w 
K
K tylko dwie ostatnie osie, pozostawiając batch na miejscu. Sprawdź także przypadek, w którym 
d
v
d 
v

  różni się od 
d
k
d 
k

 .
Odtwórz przykład z trzema pozycjami i jednostkowymi projekcjami. Porównaj otrzymane dopasowania, wagi i wyjście z przykładem w notatkach.
Dodaj maskę przyczynową. Sprawdź, czy wagi ponad przekątną są zerowe i czy wynik pierwszej pozycji jest równy jej wartości. Zmień ostatni wiersz wejścia - wyniki wcześniejszych pozycji jednej maskowanej głowy nie powinny się zmienić.
Dodaj maskę paddingu. Sprawdź zerowe wagi zamaskowanych kluczy oraz to, że zmiana wartości na tych pozycjach nie zmienia wyjść prawidłowych zapytań. Dla każdego prawidłowego wiersza suma wag powinna wynosić około 1. Nie podawaj do softmaxu wiersza bez ani jednego dozwolonego klucza.
```

```text
tool: Antigravity
model: Gemini 3.8 Flash
ok
```

```text
tool: Antigravity
model: Gemini 3.8 Flash
ok to teraz dodaj maske przyczynowa
```

```text
tool: Antigravity
model: Gemini 3.8 Flash
Wyczyscilem kod troche z niepotrzebnych printów, do czego służą te dwa? Co one mają pokazać? print("Wcześniejsze pozycje niezmienione po zmianie ostatniego wiersza:", torch.allclose(causal_output[:, :-1], causal_output_mod[:, :-1]))
print("Ostatnia pozycja zmieniona:", not torch.allclose(causal_output[:, -1], causal_output_mod[:, -1])). Btw. czy rozważyliśmy przypadek w którym Sprawdź także przypadek, w którym 
d
v
d 
v

  różni się od 
d
k
d 
k

 .
?
```

```text
tool: Antigravity
model: Gemini 3.8 Flash
1. w takim razie wyprintuj te zmienione wagi zamiast sprawdzać za mnie 2. ok
```

```text
tool: Antigravity
model: Gemini 3.8 Flash
zembeduj uzyty skill do zadania w sekcji used prompts
```
